In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import seaborn as sns
import numpy as np

# batch1 ("early"): lecanemab injections at 3–6 mo, lightsheet at 12 mo
# batch2 ("late"):  lecanemab injections at 12–15 mo, lightsheet at 15 mo
datasets = [
    {
        "name": "early",
        "participant_tsv": "/nfs/trident3/lightsheet/prado/mouse_app_lecanemab_batch2/bids/participants.tsv",
        "spimquant_dir": "/nfs/trident3/lightsheet/prado/mouse_app_lecanemab_batch2/derivatives/spimquant-v0.6.0rc2_84a605e_ozx",
    },
    {
        "name": "late",
        "participant_tsv": "/nfs/trident3/lightsheet/prado/mouse_app_lecanemab_batch3/bids/participants.tsv",
        "spimquant_dir": "/nfs/trident3/lightsheet/prado/mouse_app_lecanemab_batch3/derivatives/spimquant-v0.6.0rc2_84a605e_ozx",
    },
]


In [2]:
def load_subject_df(spimquant_dir: str, subject: str) -> pd.DataFrame | None:
    
    regionpropstats_tsv = (
        f"{spimquant_dir}/{subject}/micr/"
        f"{subject}_sample-brain_acq-imaris4x_stain-Abeta_seg-all_from-ABAv3_level-5_desc-otsu+k3i2_regionpropstats.tsv"
    )

    if not Path(regionpropstats_tsv).exists():
        return None

    
    df_subject = pd.read_csv(regionpropstats_tsv,sep="\t")
    df_subject["subject"] = subject
    return df_subject

In [3]:
dataset_dfs = {}

for dataset in datasets:
    name = dataset["name"]
    df_participants = pd.read_csv(dataset["participant_tsv"], sep="\t")

    subject_dfs = [
        df_subject
        for subject in df_participants["participant_id"]
        if (df_subject := load_subject_df(dataset["spimquant_dir"], subject)) is not None
    ]

    if not subject_dfs:
        print(f"No data found for dataset '{name}'")
        continue

    df_dataset = pd.concat(subject_dfs, ignore_index=False).merge(
        df_participants,
        left_on="subject",
        right_on="participant_id",
        how="left",
    )

    dataset_dfs[name] = df_dataset
    print(f"Loaded dataset '{name}': {len(df_dataset):,} plaques from "
          f"{df_dataset['subject'].nunique()} subjects")


Loaded dataset 'early': 613,466 plaques from 8 subjects
Loaded dataset 'late': 703,941 plaques from 10 subjects


In [4]:
voxel_vol    = 1.6 * 1.6 * 2.75           # µm³ per voxel
voxel_vol_ml = 0.0016 * 0.0016 * 0.00275   # mL per voxel

for name, df_raw in dataset_dfs.items():
    # keep only lecanemab and PBS (vehicle)
    df = df_raw.query("treatment == 'Lecanemab' or treatment == 'PBS'").copy()

    # add derived variables
    df["sdt_CD31_um"]   = df["sdt_CD31"] * 1000.0
    df["plaque_vol_um3"] = df["nvoxels"] * voxel_vol
    df["plaque_vol_ml"]  = df["nvoxels"] * voxel_vol_ml
    df["equiv_diam_um"]  = 2 * ((3 * df["plaque_vol_um3"]) / (4 * np.pi)) ** (1 / 3)

    out_path = f"data_{name}.parquet"
    df.to_parquet(out_path)
    print(f"Saved {out_path}: {len(df):,} plaques from {df['subject'].nunique()} subjects")


Saved data_early.parquet: 535,483 plaques from 6 subjects
Saved data_late.parquet: 703,941 plaques from 10 subjects
